# 04 · ADI/CV² routing + hierarchical ZINB partial pooling

Production-scale intermittency: classify each series with **ADI** and **CV²**
(Syntetos-Boylan), route smooth/erratic demand to univariate NegBin, and fit
**hierarchical ZINB** with partial pooling on `gamma0` / `beta0` by `dept_id`
(or `store_id`) for intermittent / lumpy peers in the same pool.

| Demand class | ADI / CV² | Route |
|---|---|---|
| smooth | low ADI, low CV² | univariate NegBin |
| erratic | low ADI, high CV² | univariate NegBin |
| intermittent | high ADI, low CV² | hierarchical ZINB (if ≥2 peers) |
| lumpy | high ADI, high CV² | hierarchical ZINB (if ≥2 peers) |

```bash
uv sync --group bayesian
make prep   # or cap with M5_N_SERIES / M5_LAST_N_DAYS
```

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from __future__ import annotations

import os

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from m5.config import REPO_ROOT, SETTINGS, set_global_seed
from m5.evaluation import compute_components, wrmsse
from m5.models.bayesian import (
    ADI_THRESH,
    CV2_THRESH,
    bayesian_cv,
    bayesian_routed_cv,
    classify_intermittency,
    encode_hier_panel,
    extract_posterior,
    fit_bayes_hier_zinb,
    intermittency_profiles,
    posterior_mean_forecast_hier_zinb,
    route_demand_class,
)
from m5.plots import configure_style

_pytensor_dir = REPO_ROOT / ".pytensor"
_pytensor_dir.mkdir(exist_ok=True)
os.environ.setdefault("PYTENSOR_FLAGS", f"compiledir={_pytensor_dir}")

configure_style()
set_global_seed()

HORIZON = SETTINGS.horizon
LAST_N_DAYS = min(SETTINGS.last_n_days, 200)
N_SERIES = 12
MCMC_DRAWS = 400
MCMC_TUNE = 400
POOL_COL = "dept_id"
LONG_PARQUET = SETTINGS.processed_dir / "long.parquet"

## 1 · Load panel and compute ADI / CV² profiles

In [ ]:
if not LONG_PARQUET.exists():
    raise FileNotFoundError(f"Missing {LONG_PARQUET} — run `make prep` first.")

long = pd.read_parquet(LONG_PARQUET)
long["ds"] = pd.to_datetime(long["ds"])

# Top-volume slice for fast MCMC (same cap pattern as notebook 02).
vol = long.groupby("unique_id", observed=True)["y"].sum()
hero_ids = vol.nlargest(N_SERIES).index.astype(str).tolist()
panel = (
    long.loc[long["unique_id"].astype(str).isin(hero_ids)]
    .sort_values(["unique_id", "ds"])
    .groupby("unique_id", observed=True)
    .tail(LAST_N_DAYS + HORIZON)
)

max_ds = panel["ds"].max()
cutoff = max_ds - pd.Timedelta(days=HORIZON)
train = panel.loc[panel["ds"] <= cutoff]
future = panel.loc[panel["ds"] > cutoff]

profiles = intermittency_profiles(train)
print(profiles["demand_class"].value_counts())
profiles.head(8)

## 2 · ADI vs CV² scatter (Syntetos-Boylan quadrants)

In [ ]:
plot_df = profiles.reset_index().replace([np.inf, -np.inf], np.nan).dropna(subset=["adi", "cv2"])
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(
    data=plot_df,
    x="adi",
    y="cv2",
    hue="demand_class",
    size="zero_rate",
    sizes=(30, 200),
    ax=ax,
)
ax.axvline(ADI_THRESH, ls="--", c="gray", lw=1)
ax.axhline(CV2_THRESH, ls="--", c="gray", lw=1)
ax.set_xlabel("ADI (avg demand interval)")
ax.set_ylabel("CV² (squared coef. var. on non-zero demand)")
ax.set_title(f"Intermittency map — {len(plot_df)} series")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()

## 3 · Hierarchical ZINB on intermittent/lumpy peers (partial pooling)

Pick one `dept_id` with at least two intermittent or lumpy series. The model
shrinks series-level intercepts toward the department mean:

$$
\gamma_{0,s} \sim \mathcal{N}(\mu_{\gamma, d(s)},\ \sigma_\gamma), \quad
\beta_{0,s} \sim \mathcal{N}(\mu_{\beta, d(s)},\ \sigma_\beta)
$$

Shared DOW / SNAP / event / price coefficients are estimated globally.

In [ ]:
static = train.groupby("unique_id", observed=True)[POOL_COL].first().astype(str)
profiles_pool = profiles.assign(pool=static.loc[profiles.index.astype(str)].values)
hier_classes = {"intermittent", "lumpy"}
candidates = [
    pool
    for pool, grp in profiles_pool.loc[profiles_pool["demand_class"].isin(hier_classes)].groupby("pool")
    if len(grp) >= 2
]

if not candidates:
    pool_val = static.value_counts().idxmax()
    peer_ids = static.loc[static == pool_val].index.astype(str).tolist()[:3]
    print(f"No intermittent/lumpy pool — demo fallback dept={pool_val}, peers={peer_ids}")
else:
    pool_val = candidates[0]
    peer_ids = (
        profiles_pool.loc[(profiles_pool["pool"] == pool_val) & profiles_pool["demand_class"].isin(hier_classes)]
        .index.astype(str)
        .tolist()[:4]
    )
    if len(peer_ids) < 2:
        peer_ids = static.loc[static == pool_val].index.astype(str).tolist()[:3]
    print(f"Pool {pool_val}: {peer_ids}")

hier_train = train.loc[train["unique_id"].astype(str).isin(peer_ids)]
idata, enc = fit_bayes_hier_zinb(
    hier_train,
    pool_col=POOL_COL,
    draws=MCMC_DRAWS,
    tune=MCMC_TUNE,
    chains=2,
    progressbar=True,
)
post = extract_posterior(idata, var_names=["mu_gamma0", "sigma_gamma0", "gamma0", "mu_beta0", "sigma_beta0", "beta0"])
print(f"Fitted {enc.n_series} series x {enc.n_pools} pool(s)")

In [ ]:
# Shrinkage: series intercepts vs pool means
pool_idx = enc.pool_map[pool_val]
mu_g = float(post["mu_gamma0"][:, pool_idx].mean())
mu_b = float(post["mu_beta0"][:, pool_idx].mean())
g0 = post["gamma0"].mean(axis=0)
b0 = post["beta0"].mean(axis=0)
labels = [uid for uid, _ in sorted(enc.series_map.items(), key=lambda kv: kv[1])]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(labels, g0, color="steelblue", alpha=0.8)
axes[0].axhline(mu_g, c="crimson", ls="--", label=f"pool mean={mu_g:.2f}")
axes[0].set_title("gamma0 (zero-inflation intercept)")
axes[0].tick_params(axis="x", rotation=45)
axes[0].legend(fontsize=8)

axes[1].bar(labels, b0, color="darkorange", alpha=0.8)
axes[1].axhline(mu_b, c="crimson", ls="--", label=f"pool mean={mu_b:.2f}")
axes[1].set_title("beta0 (size intercept)")
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend(fontsize=8)
plt.tight_layout()

## 4 · One-series holdout from the hierarchical fit

In [ ]:
demo_uid = peer_ids[0]
sidx = enc.series_map[demo_uid]
future_demo = future.loc[future["unique_id"].astype(str) == demo_uid]
y_hat = posterior_mean_forecast_hier_zinb(post, future_demo, series_idx=sidx)

truth = future_demo.set_index("ds")["y"]
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(truth.index, truth.values, label="actual", lw=1.5)
ax.plot(truth.index, y_hat, label="Bayes-Hier-ZINB mean", lw=1.5)
ax.set_title(f"Holdout — {demo_uid}")
ax.legend()
plt.tight_layout()

## 5 · Routed CV vs uniform NegBin / ZINB

`bayesian_routed_cv` classifies each series on the training window, caches one
hierarchical fit per `(pool, demand_class)` peer group, and falls back to
univariate models when peers are sparse.

In [ ]:
eval_long = panel.copy()
weights, scales = compute_components(eval_long)

cv_routed = bayesian_routed_cv(
    eval_long,
    h=HORIZON,
    n_windows=1,
    n_series=N_SERIES,
    pool_col=POOL_COL,
    min_hier_series=2,
    draws=MCMC_DRAWS,
    tune=MCMC_TUNE,
    chains=2,
    quiet=True,
)
cv_negbin = bayesian_cv(
    eval_long,
    h=HORIZON,
    n_windows=1,
    n_series=N_SERIES,
    likelihood="negbin",
    draws=MCMC_DRAWS,
    tune=MCMC_TUNE,
    chains=2,
    quiet=True,
)

score_routed = wrmsse(cv_routed, weights=weights, scales=scales, models=["Bayes-Routed"])
score_negbin = wrmsse(cv_negbin, weights=weights, scales=scales, models=["Bayes-NegBin"])

pd.DataFrame(
    {
        "model": ["Bayes-Routed", "Bayes-NegBin"],
        "wrmsse": [score_routed["Bayes-Routed"], score_negbin["Bayes-NegBin"]],
    }
)

In [ ]:
print("Route mix:")
print(cv_routed.groupby(["demand_class", "route"]).size().unstack(fill_value=0))

## 6 · CLI / package entry points

```python
from m5.cv import bayesian_routed_cv

cv = bayesian_routed_cv(
    long,
    n_series=15,
    pool_col="dept_id",   # or "store_id"
    min_hier_series=2,
    quiet=True,
)
```

For exploratory shrinkage on a hand-picked peer panel, use
`fit_bayes_hier_zinb` + `encode_hier_panel` directly (section 3 above).
See also the partial-pooling walkthrough in `01_hierarchical_negbin.ipynb`.